# Model 1: Legal Retrieval Reranker (All-in-One)

This single notebook does everything end-to-end:
1. Load raw Kaggle or local CSV data
2. Normalize columns
3. Create query-level train/dev/test split (if split is missing)
4. Validate data quality
5. Generate full data-preparation figures and markdown report
6. Evaluate baseline reranker
7. Fine-tune reranker
8. Evaluate on dev/test and save artifacts

In [1]:
# If needed, uncomment and run once.
# !pip install -q sentence-transformers scikit-learn pandas numpy matplotlib

from __future__ import annotations

import hashlib
import json
import math
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

c:\Users\ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def find_project_root(start: Path) -> Path:
    for current in [start, *start.parents]:
        if (current / "docs" / "ml").exists() and (current / "backend").exists():
            return current
    raise RuntimeError("Could not find project root from current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "docs" / "ml" / "data"
OUTPUT_DIR = PROJECT_ROOT / "docs" / "ml" / "outputs"
MODEL_DIR = PROJECT_ROOT / "docs" / "ml" / "models" / "reranker_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

candidate_files = [
    DATA_DIR / "reranker_pairs.csv",
    DATA_DIR / "kaggle_raw.csv",
]

DATA_FILE = next((path for path in candidate_files if path.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        "No input file found. Add one of these files: "
        f"{candidate_files[0]} or {candidate_files[1]}"
    )

print("Project root:", PROJECT_ROOT)
print("Data file:", DATA_FILE)
print("Output dir:", OUTPUT_DIR)
print("Model dir:", MODEL_DIR)

Project root: C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform
Data file: C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\data\reranker_pairs.csv
Output dir: C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\outputs
Model dir: C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\models\reranker_v1


In [3]:
df_raw = pd.read_csv(DATA_FILE)
print("Raw shape:", df_raw.shape)
print("Raw columns:", list(df_raw.columns))
df_raw.head(3)

Raw shape: (12, 9)
Raw columns: ['query_id', 'query', 'candidate_text', 'label', 'split', 'source', 'case_id', 'document_id', 'chunk_id']


,query_id,query,candidate_text,label,split,source,case_id,document_id,chunk_id
0,q_0001,What is the notice period before termination i...,The agreement requires a written notice 30 day...,1,train,eval_suite,15,101,1001
1,q_0001,What is the notice period before termination i...,Invoices must be paid within 15 days from rece...,0,train,eval_suite,15,101,1002
2,q_0001,What is the notice period before termination i...,A breach notice was sent but no termination cl...,0,train,eval_suite,15,102,1003


In [4]:
COLUMN_CANDIDATES = {
    "query_id": ["query_id", "qid", "question_id", "id"],
    "query": ["query", "question", "prompt", "user_query"],
    "candidate_text": ["candidate_text", "passage", "chunk_text", "text", "document_text"],
    "label": ["label", "relevance", "is_relevant", "target"],
    "split": ["split", "set", "subset"],
    "source": ["source", "dataset", "origin"],
    "case_id": ["case_id"],
    "document_id": ["document_id"],
    "chunk_id": ["chunk_id"],
}

def resolve_column(df: pd.DataFrame, target: str, required: bool) -> str | None:
    for candidate in COLUMN_CANDIDATES[target]:
        if candidate in df.columns:
            return candidate
    if required:
        raise ValueError(
            f"Missing required column for {target}. Acceptable names: {COLUMN_CANDIDATES[target]}"
        )
    return None

def to_binary_label(value) -> int:
    if pd.isna(value):
        return 0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return 1 if float(value) > 0 else 0

    token = str(value).strip().lower()
    positive_tokens = {"1", "true", "yes", "relevant", "support", "supported"}
    return 1 if token in positive_tokens else 0

query_col = resolve_column(df_raw, "query", required=True)
candidate_col = resolve_column(df_raw, "candidate_text", required=True)
label_col = resolve_column(df_raw, "label", required=True)
query_id_col = resolve_column(df_raw, "query_id", required=False)
split_col = resolve_column(df_raw, "split", required=False)
source_col = resolve_column(df_raw, "source", required=False)
case_id_col = resolve_column(df_raw, "case_id", required=False)
document_id_col = resolve_column(df_raw, "document_id", required=False)
chunk_id_col = resolve_column(df_raw, "chunk_id", required=False)

df = pd.DataFrame({
    "query": df_raw[query_col].astype(str).str.strip(),
    "candidate_text": df_raw[candidate_col].astype(str).str.strip(),
    "label": df_raw[label_col].apply(to_binary_label).astype(int),
})

if query_id_col:
    df["query_id"] = df_raw[query_id_col].astype(str).str.strip()
else:
    df["query_id"] = df["query"].apply(
        lambda x: "q_" + hashlib.md5(x.encode("utf-8")).hexdigest()[:10]
    )

df["source"] = df_raw[source_col].astype(str).str.strip() if source_col else "unknown"
df["case_id"] = df_raw[case_id_col] if case_id_col else None
df["document_id"] = df_raw[document_id_col] if document_id_col else None
df["chunk_id"] = df_raw[chunk_id_col] if chunk_id_col else None

if split_col:
    normalized_split = df_raw[split_col].astype(str).str.strip().str.lower()
    split_map = {"val": "dev", "valid": "dev", "validation": "dev"}
    df["split"] = normalized_split.replace(split_map)
else:
    df["split"] = ""

df = df[(df["query"] != "") & (df["candidate_text"] != "")].copy()
print("Normalized shape:", df.shape)
df.head(3)

Normalized shape: (12, 9)


,query,candidate_text,label,query_id,source,case_id,document_id,chunk_id,split
0,What is the notice period before termination i...,The agreement requires a written notice 30 day...,1,q_0001,eval_suite,15,101,1001,train
1,What is the notice period before termination i...,Invoices must be paid within 15 days from rece...,0,q_0001,eval_suite,15,101,1002,train
2,What is the notice period before termination i...,A breach notice was sent but no termination cl...,0,q_0001,eval_suite,15,102,1003,train


In [5]:
query_stats = df.groupby("query_id")["label"].agg(["count", "sum"]).rename(columns={"sum": "positives"})
query_stats["negatives"] = query_stats["count"] - query_stats["positives"]
good_query_ids = query_stats[(query_stats["positives"] > 0) & (query_stats["negatives"] > 0)].index
dropped_queries = query_stats.shape[0] - len(good_query_ids)

if dropped_queries > 0:
    print(f"Dropping {dropped_queries} query groups with only one class.")

df = df[df["query_id"].isin(good_query_ids)].copy()

if not set(df["split"].unique()).issubset({"", "train", "dev", "test"}):
    raise ValueError("split column contains unsupported values. Use train/dev/test or leave empty.")

if (df["split"] == "").all():
    unique_qids = sorted(df["query_id"].unique())
    train_qids, temp_qids = train_test_split(unique_qids, test_size=0.30, random_state=SEED)
    dev_qids, test_qids = train_test_split(temp_qids, test_size=0.50, random_state=SEED)

    train_set, dev_set, test_set = set(train_qids), set(dev_qids), set(test_qids)

    def assign_split(qid: str) -> str:
        if qid in train_set:
            return "train"
        if qid in dev_set:
            return "dev"
        return "test"

    df["split"] = df["query_id"].apply(assign_split)

split_summary = (
    df.groupby("split")
    .agg(rows=("query_id", "size"), queries=("query_id", "nunique"), positives=("label", "sum"))
    .reset_index()
)
split_summary["negatives"] = split_summary["rows"] - split_summary["positives"]
split_summary["positive_rate"] = (split_summary["positives"] / split_summary["rows"]).round(4)
split_summary

,split,rows,queries,positives,negatives,positive_rate
0,dev,3,1,1,2,0.3333
1,test,3,1,1,2,0.3333
2,train,6,2,2,4,0.3333


In [6]:
# Full data-preparation evidence: quality checks, EDA figures, and markdown report.
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass

FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def to_rel_path(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")


def save_plot(filename: str) -> str:
    plot_path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.close()
    return to_rel_path(plot_path)


df_eda = df.copy()
df_eda["query_len_chars"] = df_eda["query"].astype(str).str.len()
df_eda["candidate_len_chars"] = df_eda["candidate_text"].astype(str).str.len()
df_eda["query_len_words"] = df_eda["query"].astype(str).str.split().str.len()
df_eda["candidate_len_words"] = df_eda["candidate_text"].astype(str).str.split().str.len()

required_columns = ["query_id", "query", "candidate_text", "label", "split", "source"]
missing_rows = []
for col in required_columns:
    if col not in df_eda.columns:
        missing_rows.append({"column": col, "missing_or_empty_rows": int(df_eda.shape[0])})
        continue

    series = df_eda[col]
    if series.dtype == object:
        empty_mask = series.isna() | (series.astype(str).str.strip() == "")
    else:
        empty_mask = series.isna()

    missing_rows.append({"column": col, "missing_or_empty_rows": int(empty_mask.sum())})

missing_df = pd.DataFrame(missing_rows)

query_group = (
    df_eda.groupby("query_id")
    .agg(candidates=("candidate_text", "size"), positives=("label", "sum"))
    .reset_index()
)
query_group["negatives"] = query_group["candidates"] - query_group["positives"]
query_group["positive_rate"] = (query_group["positives"] / query_group["candidates"]).round(4)

pair_duplicates = int(df_eda.duplicated(subset=["query", "candidate_text"]).sum())
query_candidate_duplicates = int(df_eda.duplicated(subset=["query_id", "candidate_text"]).sum())
full_row_duplicates = int(df_eda.duplicated().sum())

figure_paths = []

# 1) Split row counts
split_counts = df_eda["split"].fillna("unknown").replace("", "unknown").value_counts().sort_index()
plt.figure(figsize=(8, 4))
split_counts.plot(kind="bar", color="#2b6cb0")
plt.title("Rows Per Split")
plt.xlabel("split")
plt.ylabel("rows")
figure_paths.append(save_plot("prep_01_rows_per_split.png"))

# 2) Label distribution (overall)
label_counts = df_eda["label"].value_counts().sort_index()
label_display = ["non_relevant (0)" if x == 0 else "relevant (1)" for x in label_counts.index]
plt.figure(figsize=(6, 4))
plt.bar(label_display, label_counts.values, color=["#718096", "#2f855a"])
plt.title("Overall Label Distribution")
plt.ylabel("rows")
figure_paths.append(save_plot("prep_02_label_distribution.png"))

# 3) Positive rate per split
split_pos_rate = (
    df_eda.groupby("split")["label"].mean().fillna(0).sort_index().rename("positive_rate")
)
plt.figure(figsize=(8, 4))
plt.bar(split_pos_rate.index, split_pos_rate.values, color="#dd6b20")
plt.ylim(0, 1)
plt.title("Positive Rate Per Split")
plt.xlabel("split")
plt.ylabel("positive_rate")
figure_paths.append(save_plot("prep_03_positive_rate_per_split.png"))

# 4) Query length distribution (words)
if not df_eda.empty:
    q_upper = int(df_eda["query_len_words"].quantile(0.99))
else:
    q_upper = 1
q_upper = max(q_upper, 1)
q_lengths = df_eda["query_len_words"].clip(upper=q_upper)
plt.figure(figsize=(8, 4))
plt.hist(q_lengths, bins=30, color="#4a5568", edgecolor="white")
plt.title("Query Length Distribution (Words, 99th Percentile Capped)")
plt.xlabel("query words")
plt.ylabel("count")
figure_paths.append(save_plot("prep_04_query_length_words.png"))

# 5) Candidate length distribution (words)
if not df_eda.empty:
    c_upper = int(df_eda["candidate_len_words"].quantile(0.99))
else:
    c_upper = 1
c_upper = max(c_upper, 1)
c_lengths = df_eda["candidate_len_words"].clip(upper=c_upper)
plt.figure(figsize=(8, 4))
plt.hist(c_lengths, bins=40, color="#805ad5", edgecolor="white")
plt.title("Candidate Length Distribution (Words, 99th Percentile Capped)")
plt.xlabel("candidate words")
plt.ylabel("count")
figure_paths.append(save_plot("prep_05_candidate_length_words.png"))

# 6) Candidates per query
max_candidates = int(query_group["candidates"].max()) if not query_group.empty else 1
max_candidates = max(max_candidates, 1)
plt.figure(figsize=(8, 4))
plt.hist(
    query_group["candidates"] if not query_group.empty else [0],
    bins=min(30, max_candidates),
    color="#3182ce",
    edgecolor="white",
)
plt.title("Candidates Per Query")
plt.xlabel("candidates")
plt.ylabel("queries")
figure_paths.append(save_plot("prep_06_candidates_per_query.png"))

# 7) Positives per query
max_positives = int(query_group["positives"].max()) if not query_group.empty else 1
max_positives = max(max_positives, 1)
plt.figure(figsize=(8, 4))
plt.hist(
    query_group["positives"] if not query_group.empty else [0],
    bins=min(30, max_positives + 1),
    color="#38a169",
    edgecolor="white",
)
plt.title("Positives Per Query")
plt.xlabel("positives")
plt.ylabel("queries")
figure_paths.append(save_plot("prep_07_positives_per_query.png"))

# 8) Source distribution (top 15)
source_counts = (
    df_eda["source"].fillna("unknown").replace("", "unknown").value_counts().head(15)
)
plt.figure(figsize=(9, 5))
source_counts.sort_values().plot(kind="barh", color="#2c7a7b")
plt.title("Top Data Sources")
plt.xlabel("rows")
plt.ylabel("source")
figure_paths.append(save_plot("prep_08_top_sources.png"))

# 9) Missing/empty values by key column
plt.figure(figsize=(8, 4))
plt.bar(missing_df["column"], missing_df["missing_or_empty_rows"], color="#e53e3e")
plt.title("Missing or Empty Values By Key Column")
plt.xlabel("column")
plt.ylabel("rows")
plt.xticks(rotation=30, ha="right")
figure_paths.append(save_plot("prep_09_missing_values.png"))

# 10) Duplicate diagnostics
duplicate_diag = pd.DataFrame(
    {
        "metric": [
            "duplicate (query, candidate_text)",
            "duplicate (query_id, candidate_text)",
            "duplicate full rows",
        ],
        "count": [pair_duplicates, query_candidate_duplicates, full_row_duplicates],
    }
)
plt.figure(figsize=(9, 4))
plt.bar(duplicate_diag["metric"], duplicate_diag["count"], color="#d69e2e")
plt.title("Duplicate Diagnostics")
plt.ylabel("rows")
plt.xticks(rotation=20, ha="right")
figure_paths.append(save_plot("prep_10_duplicate_diagnostics.png"))

# 11) Split by label stacked bars
split_label = pd.crosstab(df_eda["split"], df_eda["label"]).sort_index()
if 0 not in split_label.columns:
    split_label[0] = 0
if 1 not in split_label.columns:
    split_label[1] = 0
split_label = split_label[[0, 1]]
plt.figure(figsize=(8, 4))
split_label.plot(kind="bar", stacked=True, color=["#718096", "#2f855a"], ax=plt.gca())
plt.title("Split Composition By Label")
plt.xlabel("split")
plt.ylabel("rows")
plt.legend(["label=0", "label=1"])
figure_paths.append(save_plot("prep_11_split_label_composition.png"))

prep_stats = {
    "rows_total": int(df_eda.shape[0]),
    "queries_total": int(df_eda["query_id"].nunique()),
    "positive_rate_overall": round(float(df_eda["label"].mean()) if not df_eda.empty else 0.0, 6),
    "avg_candidates_per_query": round(float(query_group["candidates"].mean()) if not query_group.empty else 0.0, 4),
    "avg_positives_per_query": round(float(query_group["positives"].mean()) if not query_group.empty else 0.0, 4),
    "query_words_p50": float(df_eda["query_len_words"].quantile(0.50)) if not df_eda.empty else 0.0,
    "query_words_p95": float(df_eda["query_len_words"].quantile(0.95)) if not df_eda.empty else 0.0,
    "candidate_words_p50": float(df_eda["candidate_len_words"].quantile(0.50)) if not df_eda.empty else 0.0,
    "candidate_words_p95": float(df_eda["candidate_len_words"].quantile(0.95)) if not df_eda.empty else 0.0,
    "duplicate_query_candidate": pair_duplicates,
    "duplicate_query_id_candidate": query_candidate_duplicates,
    "duplicate_full_rows": full_row_duplicates,
    "figures": figure_paths,
}

prep_tables_payload = {
    "split_summary": split_summary.to_dict(orient="records"),
    "missing_by_column": missing_df.to_dict(orient="records"),
    "query_distribution": {
        "queries_total": int(query_group.shape[0]),
        "candidates_per_query": {
            "min": float(query_group["candidates"].min()) if not query_group.empty else 0.0,
            "median": float(query_group["candidates"].median()) if not query_group.empty else 0.0,
            "p95": float(query_group["candidates"].quantile(0.95)) if not query_group.empty else 0.0,
            "max": float(query_group["candidates"].max()) if not query_group.empty else 0.0,
        },
        "positives_per_query": {
            "min": float(query_group["positives"].min()) if not query_group.empty else 0.0,
            "median": float(query_group["positives"].median()) if not query_group.empty else 0.0,
            "p95": float(query_group["positives"].quantile(0.95)) if not query_group.empty else 0.0,
            "max": float(query_group["positives"].max()) if not query_group.empty else 0.0,
        },
    },
    "prep_stats": prep_stats,
}

prep_tables_path = OUTPUT_DIR / "prep_summary_tables.json"
with open(prep_tables_path, "w", encoding="utf-8") as f:
    json.dump(prep_tables_payload, f, ensure_ascii=False, indent=2)

prep_timestamp = datetime.now(timezone.utc).isoformat()
prep_report_path = OUTPUT_DIR / "prep_report.md"
split_summary_text = split_summary.to_string(index=False)
missing_summary_text = missing_df.to_string(index=False)

report_lines = [
    "# Reranker Data Preparation Report",
    "",
    f"Generated UTC: {prep_timestamp}",
    f"Data file: {DATA_FILE}",
    "",
    "## Core Stats",
    f"- rows_total: {prep_stats['rows_total']}",
    f"- queries_total: {prep_stats['queries_total']}",
    f"- positive_rate_overall: {prep_stats['positive_rate_overall']}",
    f"- avg_candidates_per_query: {prep_stats['avg_candidates_per_query']}",
    f"- avg_positives_per_query: {prep_stats['avg_positives_per_query']}",
    f"- duplicate_query_candidate: {prep_stats['duplicate_query_candidate']}",
    f"- duplicate_query_id_candidate: {prep_stats['duplicate_query_id_candidate']}",
    f"- duplicate_full_rows: {prep_stats['duplicate_full_rows']}",
    "",
    "## Split Summary",
    "```",
    split_summary_text,
    "```",
    "",
    "## Missing/Empty Key Fields",
    "```",
    missing_summary_text,
    "```",
    "",
    "## Figures",
]

for fig in figure_paths:
    report_lines.append(f"- {fig}")

with open(prep_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

print("Preparation artifacts saved:")
print("-", prep_report_path)
print("-", prep_tables_path)
for fig in figure_paths:
    print("-", fig)

pd.DataFrame([prep_stats])

Preparation artifacts saved:
- C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\outputs\prep_report.md
- C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\outputs\prep_summary_tables.json
- docs/ml/outputs/figures/prep_01_rows_per_split.png
- docs/ml/outputs/figures/prep_02_label_distribution.png
- docs/ml/outputs/figures/prep_03_positive_rate_per_split.png
- docs/ml/outputs/figures/prep_04_query_length_words.png
- docs/ml/outputs/figures/prep_05_candidate_length_words.png
- docs/ml/outputs/figures/prep_06_candidates_per_query.png
- docs/ml/outputs/figures/prep_07_positives_per_query.png
- docs/ml/outputs/figures/prep_08_top_sources.png
- docs/ml/outputs/figures/prep_09_missing_values.png
- docs/ml/outputs/figures/prep_10_duplicate_diagnostics.png
- docs/ml/outputs/figures/prep_11_split_label_composition.png


,rows_total,queries_total,positive_rate_overall,avg_candidates_per_query,avg_positives_per_query,query_words_p50,query_words_p95,candidate_words_p50,candidate_words_p95,duplicate_query_candidate,duplicate_query_id_candidate,duplicate_full_rows,figures
0,12,4,0.333333,3.0,1.0,7.0,10.0,9.5,11.0,0,0,0,[docs/ml/outputs/figures/prep_01_rows_per_spli...


In [7]:
def dcg_at_k(labels, k):
    labels = np.asarray(labels)[:k]
    if labels.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, labels.size + 2))
    return float(np.sum((2 ** labels - 1) / discounts))

def ndcg_at_k(labels, k):
    actual = dcg_at_k(labels, k)
    ideal = dcg_at_k(sorted(labels, reverse=True), k)
    return 0.0 if ideal == 0 else actual / ideal

def mrr(labels):
    for idx, value in enumerate(labels, start=1):
        if value == 1:
            return 1.0 / idx
    return 0.0

def recall_at_k(labels, k):
    labels = np.asarray(labels)
    total_pos = np.sum(labels == 1)
    if total_pos == 0:
        return 0.0
    return float(np.sum(labels[:k] == 1) / total_pos)

def evaluate_ranking(df_split: pd.DataFrame, scores: np.ndarray):
    temp = df_split.copy()
    temp["score"] = scores

    ndcg10_list = []
    mrr_list = []
    recall1_list = []
    recall3_list = []
    recall5_list = []

    for _, group in temp.groupby("query_id"):
        ordered = group.sort_values("score", ascending=False)
        labels = ordered["label"].tolist()
        ndcg10_list.append(ndcg_at_k(labels, 10))
        mrr_list.append(mrr(labels))
        recall1_list.append(recall_at_k(labels, 1))
        recall3_list.append(recall_at_k(labels, 3))
        recall5_list.append(recall_at_k(labels, 5))

    return {
        "queries": int(temp["query_id"].nunique()),
        "rows": int(temp.shape[0]),
        "nDCG@10": float(np.mean(ndcg10_list) if ndcg10_list else 0.0),
        "MRR": float(np.mean(mrr_list) if mrr_list else 0.0),
        "Recall@1": float(np.mean(recall1_list) if recall1_list else 0.0),
        "Recall@3": float(np.mean(recall3_list) if recall3_list else 0.0),
        "Recall@5": float(np.mean(recall5_list) if recall5_list else 0.0),
    }

In [8]:
BASE_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
baseline_model = CrossEncoder(BASE_MODEL_NAME, max_length=512)

baseline_metrics = {}
for split_name in ["dev", "test"]:
    split_df = df[df["split"] == split_name].copy()
    if split_df.empty:
        baseline_metrics[split_name] = {}
        continue

    pairs = list(zip(split_df["query"].tolist(), split_df["candidate_text"].tolist()))
    scores = baseline_model.predict(pairs, batch_size=32, show_progress_bar=True)
    baseline_metrics[split_name] = evaluate_ranking(split_df, scores)

baseline_metrics

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9544.49it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 83.32it/s]


{'dev': {'queries': 1,
  'rows': 3,
  'nDCG@10': 1.0,
  'MRR': 1.0,
  'Recall@1': 1.0,
  'Recall@3': 1.0,
  'Recall@5': 1.0},
 'test': {'queries': 1,
  'rows': 3,
  'nDCG@10': 1.0,
  'MRR': 1.0,
  'Recall@1': 1.0,
  'Recall@3': 1.0,
  'Recall@5': 1.0}}

In [11]:
train_df = df[df["split"] == "train"].copy()
dev_df = df[df["split"] == "dev"].copy()

if train_df.empty:
    raise ValueError("No training rows found. Check your split generation.")

train_samples = [
    InputExample(texts=[row.query, row.candidate_text], label=float(row.label))
    for row in train_df.itertuples(index=False)
]

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)
finetuned_model = CrossEncoder(BASE_MODEL_NAME, num_labels=1, max_length=512)

if not dev_df.empty:
    dev_labels = dev_df["label"].astype(float).tolist()
    dev_pairs = list(zip(dev_df["query"].tolist(), dev_df["candidate_text"].tolist()))

    # sentence-transformers changed evaluator signatures across versions;
    # support both (s1, s2, labels) and (pairs, labels) styles.
    try:
        evaluator = CEBinaryClassificationEvaluator(
            dev_df["query"].tolist(),
            dev_df["candidate_text"].tolist(),
            dev_labels,
            name="dev",
        )
    except TypeError:
        evaluator = CEBinaryClassificationEvaluator(
            dev_pairs,
            dev_labels,
            name="dev",
        )

    finetuned_model.fit(
        train_dataloader=train_dataloader,
        evaluator=evaluator,
        epochs=1,
        warmup_steps=max(10, len(train_dataloader) // 10),
        output_path=str(MODEL_DIR),
        save_best_model=True,
    )
else:
    finetuned_model.fit(
        train_dataloader=train_dataloader,
        epochs=1,
        warmup_steps=max(10, len(train_dataloader) // 10),
        output_path=str(MODEL_DIR),
    )

# Explicitly persist the final model because some ST versions only write eval artifacts
# when no checkpoint beats the baseline during this tiny training run.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
finetuned_model.save(str(MODEL_DIR))

print("Training complete. Model saved to:", MODEL_DIR)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5298.58it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\ahmed\AppData\Local\Temp\ipykernel_7016\1726283830.py:22: DeprecationWarning: This evaluator has been deprecated in favor of the more general CrossEncoderClassificationEvaluator. Please use CrossEncoderClassificationEvaluator instead, which supports both binary and multi-class evaluation. It accepts approximately the same inputs as this evaluator.
  evaluator = CEBinaryClassificationEvaluator(
C:\Users\ahmed\AppData\Local\Temp\ipykernel_7016\1726283830.py:29: DeprecationWarning: This evaluator has been deprecated in favor of the more general CrossEncoderClassificati

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.24it/s]

Training complete. Model saved to: C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\models\reranker_v1


In [12]:
trained_model = CrossEncoder(str(MODEL_DIR), max_length=512)
trained_metrics = {}

for split_name in ["dev", "test"]:
    split_df = df[df["split"] == split_name].copy()
    if split_df.empty:
        trained_metrics[split_name] = {}
        continue

    pairs = list(zip(split_df["query"].tolist(), split_df["candidate_text"].tolist()))
    scores = trained_model.predict(pairs, batch_size=32, show_progress_bar=True)
    trained_metrics[split_name] = evaluate_ranking(split_df, scores)

rows = []
for split_name in ["dev", "test"]:
    base = baseline_metrics.get(split_name, {})
    tuned = trained_metrics.get(split_name, {})
    for metric_name in ["nDCG@10", "MRR", "Recall@1", "Recall@3", "Recall@5"]:
        base_val = float(base.get(metric_name, 0.0))
        tuned_val = float(tuned.get(metric_name, 0.0))
        rows.append({
            "split": split_name,
            "metric": metric_name,
            "baseline": round(base_val, 6),
            "trained": round(tuned_val, 6),
            "delta": round(tuned_val - base_val, 6),
        })

comparison_df = pd.DataFrame(rows)
comparison_df

Batches: 100%|██████████| 1/1 [00:00<00:00, 43.35it/s]


,split,metric,baseline,trained,delta
0,dev,nDCG@10,1.0,1.0,0.0
1,dev,MRR,1.0,1.0,0.0
2,dev,Recall@1,1.0,1.0,0.0
3,dev,Recall@3,1.0,1.0,0.0
4,dev,Recall@5,1.0,1.0,0.0
5,test,nDCG@10,1.0,1.0,0.0
6,test,MRR,1.0,1.0,0.0
7,test,Recall@1,1.0,1.0,0.0
8,test,Recall@3,1.0,1.0,0.0
9,test,Recall@5,1.0,1.0,0.0


In [13]:
timestamp = datetime.now(timezone.utc).isoformat()


def _safe_rel(path_obj):
    if path_obj is None:
        return None
    try:
        return str(Path(path_obj).relative_to(PROJECT_ROOT)).replace("\\", "/")
    except Exception:
        return str(path_obj)


prep_stats_payload = prep_stats if "prep_stats" in globals() else {}
prep_report_rel = _safe_rel(prep_report_path) if "prep_report_path" in globals() else None
prep_tables_rel = _safe_rel(prep_tables_path) if "prep_tables_path" in globals() else None

payload = {
    "generated_at_utc": timestamp,
    "data_file": str(DATA_FILE),
    "rows_total": int(df.shape[0]),
    "queries_total": int(df["query_id"].nunique()),
    "split_counts": (
        df.groupby("split").agg(rows=("query_id", "size"), queries=("query_id", "nunique")).reset_index().to_dict(orient="records")
    ),
    "prep_stats": prep_stats_payload,
    "prep_report_markdown": prep_report_rel,
    "prep_tables_json": prep_tables_rel,
    "baseline": baseline_metrics,
    "trained": trained_metrics,
    "comparison": comparison_df.to_dict(orient="records"),
}

metrics_json_path = OUTPUT_DIR / "reranker_metrics.json"
metrics_csv_path = OUTPUT_DIR / "reranker_metrics.csv"
prepared_csv_path = OUTPUT_DIR / "reranker_pairs_prepared.csv"

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

comparison_df.to_csv(metrics_csv_path, index=False)
df.to_csv(prepared_csv_path, index=False)

print("Saved:")
print("-", metrics_json_path)
print("-", metrics_csv_path)
print("-", prepared_csv_path)
if prep_report_rel:
    print("-", prep_report_rel)
if prep_tables_rel:
    print("-", prep_tables_rel)
print("-", MODEL_DIR)

Saved:
- C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\outputs\reranker_metrics.json
- C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\outputs\reranker_metrics.csv
- C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\outputs\reranker_pairs_prepared.csv
- docs/ml/outputs/prep_report.md
- docs/ml/outputs/prep_summary_tables.json
- C:\Users\ahmed\Desktop\pfe.2\legal-ai-platform\docs\ml\models\reranker_v1


## What data you should bring next

Drop one file at docs/ml/data/reranker_pairs.csv or docs/ml/data/kaggle_raw.csv with at least:
- query (or question)
- candidate_text (or passage/text/chunk_text)
- label (0 or 1, or relevance that can map to 0/1)

Optional columns: query_id, split, source, case_id, document_id, chunk_id.
If split is missing, this notebook will create train/dev/test by query_id automatically.